# Run the 3D model — see it segment a real scan

The training and evaluation notebooks produce numbers. This one produces
**pictures**: it loads the trained 3D U-Net, segments complete brain volumes,
and shows the prediction beside the radiologist's own annotation.

**Why this runs on Kaggle and not on the laptop.** Whole-volume 3D inference
needs roughly 15 GB of VRAM for the sliding window. The development machine has
6 GB, which is the same constraint that forced 2D in the first place. A Kaggle
T4 has enough, and the data is already mounted here — so this is where the model
can actually be exercised.

**Every case shown is one the model never trained on.** The split is rebuilt
from the checkpoint's own stored seed, so there is no way to accidentally show a
training patient and flatter the result.

---

### Setup

*Add Data* → the BraTS dataset (`awsaf49/brats20-dataset-training-validation`),
and *Your Work → Notebook Output* → the training kernel, so `best.pt` is
available. Turn the accelerator on.

In [ ]:
import os, glob, json, random
import numpy as np, torch, nibabel as nib
import matplotlib.pyplot as plt

CKPT, ROOT = '', ''
if not CKPT:
    for pat in ('/kaggle/input/**/best.pt', '/kaggle/working/checkpoints/best.pt',
                '/content/drive/MyDrive/mri_3d_unet/best.pt', './checkpoints/best.pt'):
        hit = glob.glob(pat, recursive=True)
        if hit: CKPT = hit[0]; break
if not ROOT:
    hit = (glob.glob('/kaggle/input/**/MICCAI_BraTS2020_TrainingData', recursive=True)
           or glob.glob('/content/data/MICCAI_BraTS2020_TrainingData'))
    ROOT = hit[0] if hit else ''
assert CKPT and ROOT, "attach the training kernel output and the BraTS dataset"

# Kaggle hands out either a T4 (sm_75) or a P100 (sm_60); their PyTorch build
# dropped sm_60, so check before trusting CUDA rather than crashing on it.
dev = 'cpu'
if torch.cuda.is_available():
    mj, mn = torch.cuda.get_device_capability(0)
    if f'sm_{mj}{mn}' in torch.cuda.get_arch_list():
        dev = 'cuda'
    else:
        print(f"GPU is sm_{mj}{mn}, unsupported by this PyTorch — using CPU (slow)")

ck = torch.load(CKPT, map_location=dev)
cfg = ck.get('config', {})
print(f"checkpoint : epoch {ck['epoch']}  (patch-based Dice {ck.get('best_dice',0):.4f})")
print(f"device     : {dev}")
print(f"data       : {ROOT}")

## Model and the held-out patients

Split rebuilt from the checkpoint's own config — same seed, same case count — so
these are exactly the patients excluded from training.

In [ ]:
import torch.nn as nn, torch.nn.functional as F

NUM_CLASSES, MODALITIES = 4, ['t1', 't1ce', 't2', 'flair']
PATCH     = cfg.get('patch', 128)
BASE_FILT = cfg.get('base_filters', 16)
N_CASES   = cfg.get('n_cases', 126)
SEED      = cfg.get('seed', 42)

def block(i, o):
    return nn.Sequential(
        nn.Conv3d(i, o, 3, padding=1, bias=False), nn.InstanceNorm3d(o), nn.LeakyReLU(0.01, True),
        nn.Conv3d(o, o, 3, padding=1, bias=False), nn.InstanceNorm3d(o), nn.LeakyReLU(0.01, True))

class UNet3D(nn.Module):
    def __init__(self, in_ch=4, n_cls=NUM_CLASSES, f=BASE_FILT):
        super().__init__()
        self.e1, self.e2, self.e3 = block(in_ch, f), block(f, f*2), block(f*2, f*4)
        self.bott = block(f*4, f*8)
        self.u3 = nn.ConvTranspose3d(f*8, f*4, 2, 2); self.d3 = block(f*8, f*4)
        self.u2 = nn.ConvTranspose3d(f*4, f*2, 2, 2); self.d2 = block(f*4, f*2)
        self.u1 = nn.ConvTranspose3d(f*2, f,   2, 2); self.d1 = block(f*2, f)
        self.out = nn.Conv3d(f, n_cls, 1); self.pool = nn.MaxPool3d(2)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        b = self.bott(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(b),  e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

model = UNet3D().to(dev); model.load_state_dict(ck['model']); model.eval()

cases = sorted(d for d in os.listdir(ROOT) if d.startswith('BraTS20'))
dirs = [os.path.join(ROOT, c) for c in cases[:N_CASES]]
dirs = [d for d in dirs if os.path.exists(f"{d}/{os.path.basename(d)}_seg.nii")]
random.Random(SEED).shuffle(dirs)
val_dirs = dirs[:max(1, int(len(dirs) * 0.2))]
print(f"{len(val_dirs)} held-out patients available:")
for d in val_dirs[:8]: print("  ", os.path.basename(d))

## Predict a whole volume

Sliding window at 50 % overlap, logits averaged before the argmax. Overlap
matters because a patch's edge predictions have no surrounding context to draw
on; averaging means every voxel is decided partly from nearer a patch centre.

In [ ]:
def norm(v):
    b = v[v > 0]
    if b.size == 0: return v.astype(np.float32)
    return np.clip((v - b.mean()) / (b.std() + 1e-8), -5, 5).astype(np.float32)

@torch.no_grad()
def predict(x, patch=PATCH):
    stride = patch // 2 if dev == 'cuda' else patch
    _, H, W, D = x.shape
    logits = torch.zeros((NUM_CLASSES, H, W, D), device=dev)
    counts = torch.zeros((1, H, W, D), device=dev)
    def starts(n):
        if n <= patch: return [0]
        s = list(range(0, n - patch + 1, stride))
        if s[-1] != n - patch: s.append(n - patch)
        return s
    for i in starts(H):
        for j in starts(W):
            for k in starts(D):
                sl = (slice(None), slice(i,i+patch), slice(j,j+patch), slice(k,k+patch))
                t = x[sl].unsqueeze(0)
                pad = [0, max(patch-t.shape[4],0), 0, max(patch-t.shape[3],0),
                       0, max(patch-t.shape[2],0)]
                if any(pad): t = F.pad(t, pad)
                with torch.amp.autocast('cuda', enabled=(dev=='cuda')):
                    o = model(t)[0].float()
                o = o[:, :sl[1].stop-sl[1].start, :sl[2].stop-sl[2].start,
                        :sl[3].stop-sl[3].start]
                logits[:, sl[1], sl[2], sl[3]] += o
                counts[:, sl[1], sl[2], sl[3]] += 1
    return (logits / counts.clamp(min=1)).argmax(0).cpu().numpy()

def load_case(d):
    cid = os.path.basename(d)
    x = np.stack([norm(nib.load(f"{d}/{cid}_{m}.nii").get_fdata()) for m in MODALITIES])
    gt = nib.load(f"{d}/{cid}_seg.nii").get_fdata().astype(np.int64); gt[gt == 4] = 3
    return cid, x, gt

def dice(p, g, c):
    pm, gm = (p == c), (g == c)
    s = pm.sum() + gm.sum()
    return float(2 * (pm & gm).sum() / s) if s else float('nan')

print("ready")

## Show it

For each patient: the FLAIR scan, our prediction, and the radiologist's
annotation, on the slice containing the most tumour.

Colours — <span style="color:#d33">red</span> enhancing tumour,
<span style="color:#2a2">green</span> oedema,
<span style="color:#36c">blue</span> necrotic core.

Per-class Dice is printed under each case, so you can see where it agrees and
where it does not rather than only reading an average.

In [ ]:
N_SHOW = 4          # raise for more patients (~20-40 s each on GPU)
CLASSES = ['background', 'necrotic', 'oedema', 'enhancing']
COLOURS = {1: (0.2, 0.4, 0.9), 2: (0.2, 0.7, 0.2), 3: (0.9, 0.2, 0.2)}

def overlay(bg, lab):
    rgb = np.stack([bg]*3, -1)
    for c, col in COLOURS.items():
        m = lab == c
        for k in range(3):
            rgb[..., k][m] = 0.35*rgb[..., k][m] + 0.65*col[k]
    return np.clip(rgb, 0, 1)

rows, scores = [], []
for d in val_dirs[:N_SHOW]:
    cid, x, gt = load_case(d)
    pred = predict(torch.from_numpy(x).to(dev))
    torch.cuda.empty_cache()

    z = int(np.argmax([(gt[..., k] > 0).sum() for k in range(gt.shape[2])]))
    flair = x[3][..., z]; flair = (flair - flair.min()) / (np.ptp(flair) + 1e-8)
    per = {CLASSES[c]: dice(pred, gt, c) for c in (1, 2, 3)}
    per['mean'] = float(np.nanmean(list(per.values())))
    scores.append({'case': cid, **per})
    rows.append((cid, flair, overlay(flair, pred[..., z]), overlay(flair, gt[..., z]), per))
    print(f"{cid}  mean {per['mean']:.3f}   "
          + "  ".join(f"{k} {v:.3f}" for k, v in per.items() if k != 'mean'), flush=True)

fig, axes = plt.subplots(len(rows), 3, figsize=(11, 3.7*len(rows)))
axes = np.atleast_2d(axes)
for r, (cid, f_, p_, g_, per) in enumerate(rows):
    for c, (im, ttl) in enumerate([(f_, 'FLAIR (input)'),
                                   (p_, 'our 3D model'),
                                   (g_, 'radiologist')]):
        axes[r, c].imshow(np.rot90(im), cmap='gray' if c == 0 else None)
        axes[r, c].set_title(ttl if r == 0 else '', fontsize=10)
        axes[r, c].axis('off')
    # Dice belongs on the ROW, not the column header — each patient has its own,
    # and a single score above every row reads as if it applied to all of them.
    axes[r, 1].text(0.5, -0.035, f"Dice {per['mean']:.3f}", fontsize=9,
                    transform=axes[r, 1].transAxes, ha='center',
                    color='#1f6f4a' if per['mean'] >= 0.8 else '#8a6d1f')
    axes[r, 0].set_ylabel(cid, fontsize=8)
    axes[r, 0].text(-0.04, 0.5, cid, transform=axes[r, 0].transAxes,
                    rotation=90, va='center', ha='right', fontsize=8)
plt.tight_layout(); plt.savefig('/kaggle/working/predictions.png', dpi=130)
plt.show()

m = float(np.nanmean([s['mean'] for s in scores]))
print(f"\nmean over the {len(scores)} shown: {m:.4f}   (2D baseline 0.76)")
json.dump(scores, open('/kaggle/working/shown_cases.json', 'w'), indent=2)

## What to take from this

The middle column is our model, the right is the radiologist. On a good case
they are close to indistinguishable at a glance — but look at the boundaries and
at the necrotic core, which is where models usually disagree with the human.

**Do not read a single case as the result.** These are four patients out of
twenty-five; the honest figure is the whole-volume evaluation across all of
them, **Dice 0.827**, against the 2D model's **0.760**.

**And what this does not show:** a case where the model fails badly. If you want
that, raise `N_SHOW` and look for the lowest-scoring patient — a model that is
never shown failing has not really been shown.